# COMPX326 Assignment 1
## Convolutional Kernels (TensorFlow / Keras Version)

---

## Abstract

In this assignment, you will implement convolutional kernels **from scratch using NumPy** and then connect that understanding to **TensorFlow/Keras** by performing one forward pass and one backward pass with automatic differentiation.

The purpose of this assignment is to help you understand convolution at two levels:

1. **Low-level implementation** using NumPy slicing, element-wise multiplication, and summation.
2. **Deep learning framework usage** using TensorFlow/Keras for forward propagation, loss computation, gradient computation, and parameter updates.

By the end of this assignment, you should be able to:

- explain how a convolution kernel slides over an input
- implement valid convolution in 1D and 2D
- handle multiple input and output channels
- connect manual convolution to TensorFlow's tensor-based implementation
- inspect gradients and understand one step of gradient descent


## Submission Format

- Complete this assignment in a **Jupyter Notebook (`.ipynb`)**.
- Keep the function names exactly as provided.
- You may add small helper code cells for your own experiments, but your final notebook should clearly present the required functions and tests.
- Write code that you can explain confidently line by line.

This notebook version replaces the `.py` workflow. Instead of uncommenting test blocks, you should run the test cells provided after each task.


## Allowed Libraries

You may use:

- `numpy`
- `tensorflow`
- `keras` components accessed through TensorFlow, e.g. `tf.keras.optimizers.SGD`

### Important restriction

For **Tasks 1-4**, do **not** use high-level convolution functions such as:

- `tf.nn.conv2d`
- `tf.keras.layers.Conv1D`
- `tf.keras.layers.Conv2D`

Tasks 1-4 must be implemented manually with NumPy.

For **Task 5**, you should use TensorFlow for forward and backward propagation.


## Assignment Structure

You will complete five tasks:

1. `conv1d_one_in_one_out()`
2. `conv2d_one_in_one_out()`
3. `conv2d_multiple_in_one_out()`
4. `conv2d_multiple_in_multiple_out()`
5. Forward and backward propagation in TensorFlow/Keras

Functions written earlier may help with later tasks. Reuse your earlier functions whenever appropriate.


In [1]:
import numpy as np
import tensorflow as tf


2026-03-26 12:24:17.275367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774527857.571554      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774527857.650863      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774527858.368366      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774527858.368442      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774527858.368445      17 computation_placer.cc:177] computation placer alr

# Task 1: `conv1d_one_in_one_out()`

## Description

In this task, you will implement **1D valid convolution** for:

- one input feature map
- one kernel
- one output feature map

The input and kernel are both 1D NumPy arrays.

If the input has length `N` and the kernel has length `K`, then the output has length:

`N - K + 1`

At each output position, you should:

1. take a slice of the input of length `K`
2. multiply it element-wise with the kernel
3. sum the result

This is the same sliding-window idea used in CNNs, but in one dimension.


## Notes

- Use NumPy slicing: `x[i:i+k]`
- Use element-wise multiplication: `window * kernel`
- Use `np.sum(...)`
- Return a NumPy array, not a Python list


In [2]:
def conv1d_one_in_one_out(x, kernel):
    """
    Perform 1D valid convolution with one input channel and one output channel.

    Parameters
    ----------
    x : np.ndarray
        1D input feature map.
    kernel : np.ndarray
        1D convolution kernel.

    Returns
    -------
    np.ndarray
        1D output feature map.
    """
    x = np.asarray(x)
    kernel = np.asarray(kernel)

    k = kernel.shape[0]
    output_length = x.shape[0] - k + 1
    output = []

    # TODO: implement valid convolution
    for i in range(output_length):
        window = [x[i + j] * kernel[j] for j in range(0, 3)]
        # print(window)
        value = np.sum(window)
        # print(value)
        output.append(value)

    return np.asarray(output)


## Suggested test for Task 1

Run the following cell after completing the function.


In [3]:
x = np.array([1, 2, 3, 4, 5])
kernel = np.array([1, 0, -1])

print("Input:", x)
print("Kernel:", kernel)
print("Output:", conv1d_one_in_one_out(x, kernel))


Input: [1 2 3 4 5]
Kernel: [ 1  0 -1]
Output: [-2 -2 -2]


# Task 2: `conv2d_one_in_one_out()`

## Description

In this task, you will implement **2D valid convolution** for:

- one input channel
- one kernel
- one output channel

Now the input is a 2D array and the kernel is a 2D array.

If the input has shape `(H, W)` and the kernel has shape `(kH, kW)`, then the output has shape:

`(H - kH + 1, W - kW + 1)`

At each output position:

1. take a 2D slice of shape `(kH, kW)` from the input
2. multiply it element-wise with the kernel
3. sum all resulting values

This is the direct 2D extension of Task 1.


## Notes

Useful slicing pattern:

- `x[i:i+kH, j:j+kW]`

You may find it convenient to build the output row by row.


In [4]:
def conv2d_one_in_one_out(x, kernel):
    """
    Perform 2D valid convolution with one input channel and one output channel.

    Parameters
    ----------
    x : np.ndarray
        2D input feature map of shape (H, W).
    kernel : np.ndarray
        2D kernel of shape (kH, kW).

    Returns
    -------
    np.ndarray
        2D output feature map.
    """
    x = np.asarray(x)
    kernel = np.asarray(kernel)

    H, W = x.shape
    kH, kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = []

    # TODO: implement valid 2D convolution
    for i in range(out_H):
        row = []
        for j in range(out_W):
            value = 0
            for m in range(kW):
                for n in range(kH):
                    value += x[i + m, j + n] * kernel[m, n]

            row.append(value)
        output.append(row)

    return np.asarray(output)


## Suggested test for Task 2

Use a small example so you can manually inspect the result.


In [5]:
x = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

kernel = np.array([
    [1, 0],
    [0, -1]
])

print("Input:\n", x)
print("Kernel:\n", kernel)
print("Output:\n", conv2d_one_in_one_out(x, kernel))


Input:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Kernel:
 [[ 1  0]
 [ 0 -1]]
Output:
 [[-4 -4]
 [-4 -4]]


# Task 3: `conv2d_multiple_in_one_out()`

## Description

In this task, you will implement **2D convolution with multiple input channels and one output channel**.

The input now has shape:

`(C, H, W)`

where:

- `C` = number of input channels
- `H` = height
- `W` = width

The kernel has shape:

`(C, kH, kW)`

This means the kernel contains one 2D filter for each input channel.

At each output position:

1. extract a patch of shape `(C, kH, kW)` from the input
2. multiply it element-wise with the kernel
3. sum across **all channels and spatial positions**

The result is still a **single output feature map**.


## Notes

Useful slicing pattern:

- `x[:, i:i+kH, j:j+kW]`

The colon `:` keeps all channels.

This task introduces the important CNN idea that one output channel is produced by combining information across **all input channels**.


In [ ]:
def conv2d_multiple_in_one_out(x, kernel):
    """
    Perform 2D valid convolution with multiple input channels and one output channel.

    Parameters
    ----------
    x : np.ndarray
        3D input feature map of shape (C, H, W).
    kernel : np.ndarray
        3D kernel of shape (C, kH, kW).

    Returns
    -------
    np.ndarray
        2D output feature map.
    """
    x = np.asarray(x)
    kernel = np.asarray(kernel)

    C, H, W = x.shape
    Ck, kH, kW = kernel.shape
    assert C == Ck, "Kernel must have the same number of channels as the input."

    out_H = H - kH + 1
    out_W = W - kW + 1
    output = []

    # TODO: implement multi-channel to single-channel convolution
    for i in range(out_H):
        row = []
        for j in range(out_W):
            tmp = np.zeros((C, kH, kW))
            for k in range(C):
                tmp += np.multiply(x[k, i : i + kH, j : j + kW], kernel[k, i : i + kH, j : j + kW])
            
            value = np.sum(tmp)
            
            row.append(value)
        output.append(row)

    return np.asarray(output)


## Suggested test for Task 3

A random test is useful for checking shapes.


In [7]:
np.random.seed(0)
x = np.random.randn(2, 4, 4)
kernel = np.random.randn(2, 3, 3)

out = conv2d_multiple_in_one_out(x, kernel)
print("Input shape:", x.shape)
print("Kernel shape:", kernel.shape)
print("Output shape:", out.shape)
print(out)


# Task 4: `conv2d_multiple_in_multiple_out()`

## Description

In this task, you will implement **2D convolution with multiple input channels and multiple output channels**.

The input has shape:

`(C_in, H, W)`

The kernels have shape:

`(C_out, C_in, kH, kW)`

where:

- `C_out` is the number of output channels
- each output channel has its own kernel
- each kernel spans all input channels

Each kernel produces one output feature map.

So the output has shape:

`(C_out, H - kH + 1, W - kW + 1)`

This task is very close to how a real convolution layer works.


## Recommended approach

Reuse your function from Task 3.

For each output kernel:

1. select one kernel of shape `(C_in, kH, kW)`
2. apply `conv2d_multiple_in_one_out()`
3. append the result to a list

At the end, convert the list to a NumPy array.


In [ ]:
def conv2d_multiple_in_multiple_out(x, kernels):
    """
    Perform 2D valid convolution with multiple input channels and multiple output channels.

    Parameters
    ----------
    x : np.ndarray
        3D input feature map of shape (C_in, H, W).
    kernels : np.ndarray
        4D kernel tensor of shape (C_out, C_in, kH, kW).

    Returns
    -------
    np.ndarray
        3D output feature map of shape (C_out, H_out, W_out).
    """
    x = np.asarray(x)
    kernels = np.asarray(kernels)

    outputs = []

    # TODO: loop over output kernels and reuse Task 3
    # for kernel_idx in range(kernels.shape[0]):
    #     kernel = ...
    #     out = ...
    #     outputs.append(out)

    return np.asarray(outputs)


## Suggested test for Task 4

This test checks the full multi-input, multi-output case.


In [ ]:
np.random.seed(1)
x = np.random.randn(3, 5, 5)
kernels = np.random.randn(4, 3, 3, 3)

out = conv2d_multiple_in_multiple_out(x, kernels)
print("Input shape:", x.shape)
print("Kernels shape:", kernels.shape)
print("Output shape:", out.shape)


# Task 5: Forward and Backward Propagation in TensorFlow / Keras

## Description

In this task, you will move from manual NumPy convolution to TensorFlow.

You will:

1. perform a forward pass using TensorFlow convolution
2. compute an L2 loss
3. compute gradients with respect to the kernel
4. apply one SGD update step
5. compare the TensorFlow result with your NumPy implementation from Task 4

This task is designed to show how modern deep learning frameworks automate gradient computation while still following the same mathematical ideas.


## Tensor shape note

Your NumPy implementation in Task 4 uses channel-first format:

- input: `(C_in, H, W)`
- kernels: `(C_out, C_in, kH, kW)`

TensorFlow's `tf.nn.conv2d` expects channel-last format:

- input: `(batch, H, W, C_in)`
- kernel: `(kH, kW, C_in, C_out)`

So you must carefully reshape or transpose arrays when comparing NumPy and TensorFlow results.


## What you should implement

Inside a `tf.GradientTape()` block:

1. compute `y_pred` using `tf.nn.conv2d(...)`
2. compute L2 loss using:
   - element-wise difference
   - element-wise square
   - `tf.reduce_sum(...)`

After that:

3. compute gradients with `tape.gradient(...)`
4. apply gradients using `tf.keras.optimizers.SGD`

You should also print:

- the predicted output shape
- the loss value
- the gradient shape
- part of the kernel before and after update


In [ ]:
# Example data for Task 5
np.random.seed(2)
tf.random.set_seed(2)

# NumPy-style data for comparison with Task 4
x_np = np.random.randn(3, 5, 5).astype(np.float32)              # (C_in, H, W)
kernels_np = np.random.randn(2, 3, 3, 3).astype(np.float32)    # (C_out, C_in, kH, kW)

# Manual NumPy output from Task 4
y_np = conv2d_multiple_in_multiple_out(x_np, kernels_np)

# Convert input to TensorFlow channel-last format
x_tf = tf.constant(np.transpose(x_np, (1, 2, 0))[None, ...])   # (1, H, W, C_in)

# Convert kernels to TensorFlow format: (kH, kW, C_in, C_out)
kernel_tf_init = np.transpose(kernels_np, (2, 3, 1, 0))
kernel_tf = tf.Variable(kernel_tf_init)

# Create a target tensor with the same shape as TensorFlow output
y_true = tf.random.normal((1, 3, 3, 2))

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)


## Starter code for Task 5

Complete the TODO parts below.


In [ ]:
kernel_before = kernel_tf.numpy().copy()

with tf.GradientTape() as tape:
    # TODO: forward propagation
    # y_pred = ...

    # TODO: compute L2 loss
    # loss = ...
    pass

# TODO: compute gradients with respect to kernel_tf
# grads = ...

# TODO: update kernel using optimizer
# optimizer.apply_gradients(...)

# TODO: print useful debugging information
# print("Loss:", ...)
# print("Gradient shape:", ...)
# print("Kernel changed:", ...)


## Possible test for Task 5

The following test cell shows one complete version of the forward and backward pass. Use it to check your implementation after you finish the starter cell.


In [ ]:
kernel_tf_test = tf.Variable(kernel_tf_init.copy())
optimizer_test = tf.keras.optimizers.SGD(learning_rate=0.001)

kernel_before_test = kernel_tf_test.numpy().copy()

with tf.GradientTape() as tape:
    y_pred_tf = tf.nn.conv2d(x_tf, kernel_tf_test, strides=1, padding='VALID')
    loss_tf = tf.reduce_sum((y_pred_tf - y_true) ** 2)

grads_tf = tape.gradient(loss_tf, [kernel_tf_test])
optimizer_test.apply_gradients(zip(grads_tf, [kernel_tf_test]))

print("Predicted output shape:", y_pred_tf.shape)
print("Loss:", float(loss_tf.numpy()))
print("Gradient shape:", grads_tf[0].shape)
print("Kernel changed:", not np.allclose(kernel_before_test, kernel_tf_test.numpy()))

# Compare TensorFlow output with NumPy output from Task 4
y_pred_tf_np = np.transpose(y_pred_tf.numpy()[0], (2, 0, 1))  # back to (C_out, H_out, W_out)
print("NumPy output shape:", y_np.shape)
print("TensorFlow output shape converted back:", y_pred_tf_np.shape)
print("NumPy vs TensorFlow close:", np.allclose(y_np, y_pred_tf_np, atol=1e-5))


## What to verify in Task 5

After running your code, check the following:

1. The TensorFlow output should match the output from your NumPy implementation in Task 4, apart from very small floating-point differences.
2. The gradient should have the same shape as the TensorFlow kernel.
3. The kernel values should change after one SGD step.
4. The update should follow gradient descent:

`new_kernel = old_kernel - learning_rate * gradient`


# Optional Reflection Questions

Write short answers in markdown cells below your code.

1. Why should the TensorFlow output match your NumPy implementation?
2. In Task 3, why does one output channel depend on all input channels?
3. What does the gradient with respect to the kernel represent?
4. Why does subtracting the gradient move the kernel in a better direction?
5. What is the main difference between implementing convolution manually and using TensorFlow?


# Final Checklist

Before submitting your notebook, make sure:

- all required functions are present
- function names are unchanged
- all code cells run without errors
- your Task 5 TensorFlow code computes gradients and updates the kernel
- you have tested your functions with the provided examples
- your notebook is clearly organised and readable
